In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

import itertools
from collections import defaultdict

from scripts.ss_graphgen import *

In [2]:
dttz = pd.read_csv("../data/employee_productivity.csv")
dtz = dttz.drop_duplicates().reset_index(drop=True)

print("Original number of rows: ", len(dtz))
print(" ")

seln = 500

dtx = dtz.sample(n=seln, replace=False, random_state=42).reset_index(drop=True)
dt = dtx.drop_duplicates().reset_index(drop=True)
dt


Original number of rows:  1035
 


,quarter,department,team,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,target
0,2,2,7,22.94,1450.0,10080,30,0.0,0,0,56.0,1
1,4,1,12,4.08,0.0,1080,0,0.0,0,0,9.0,0
2,1,2,5,30.10,610.0,7080,56,0.0,0,0,59.0,1
3,1,1,11,4.15,0.0,1440,0,0.0,0,0,8.0,1
4,0,2,4,30.10,287.0,6060,23,150.0,15,0,55.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...
495,4,1,12,4.08,0.0,1080,0,0.0,0,0,9.0,1
496,1,2,11,10.05,13.0,3240,34,0.0,0,0,54.0,0
497,0,2,11,20.10,1192.0,6120,75,0.0,0,0,54.0,1
498,0,0,2,3.90,0.0,1200,0,0.0,0,0,10.0,1


In [3]:
target_column = "target"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 

s = dt["department"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)



In [4]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

s_LHS_random = s[lhs_idx_negative]
s_RHS_random = s[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(112, 450, 50, 112, 338)

In [5]:
##########
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                 XRHS=X_RHS_random, 
                                                                 yRHS=y_RHS_random, 
                                                                 sLHS=s_LHS_random, 
                                                                 sRHS=s_RHS_random,                                                                  
                                                                 k_min=1, 
                                                                 k_max=2)

r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]:  
    print(f"{agent}: {targets}")
    
    
print("positive ", list({t for t, l in r_labels_random.items() if l == 1})[:10])
print("negative ", list({t for t, l in r_labels_random.items() if l == -1})[:10])
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: ['t_11 (-)', 't_38 (+)']
x_1: ['t_2 (+)', 't_30 (+)']
x_2: ['t_36 (+)', 't_13 (-)']
x_3: ['t_16 (+)', 't_34 (-)']
x_4: ['t_29 (-)', 't_16 (+)']
x_5: ['t_18 (-)', 't_35 (+)']
x_6: ['t_26 (+)', 't_10 (+)']
x_7: ['t_38 (+)', 't_12 (+)']
x_8: ['t_20 (+)', 't_38 (+)']
x_9: ['t_18 (-)', 't_46 (-)']
positive  [1, 2, 4, 5, 6, 9, 10, 12, 14, 16]
negative  [34, 7, 8, 11, 44, 13, 46, 18, 23, 28]


(112, 44)

In [6]:
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                    XRHS=X_RHS_random, 
                                                                    yRHS=y_RHS_random, 
                                                                    sLHS=s_LHS_random, 
                                                                    sRHS=s_RHS_random,                                                               
                                                                    threshold=0.2, 
                                                                    topk=False, 
                                                                    thresh=True)


r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print("positive ", list({t for t, l in r_labels_random.items() if l == 1})[:10])
print("negative ", list({t for t, l in r_labels_random.items() if l == -1})[:10])
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: []
x_1: []
x_2: []
x_3: []
x_4: []
x_5: []
x_6: []
x_7: []
x_8: []
x_9: []
positive  [21]
negative  [18, 34, 44]


(112, 4)

## General

In [7]:
#############
##knn##
#############
graphsRandom1 = []
graphsinfor = []
graphid = 0
for kmax1 in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    _, _, graphx_random1 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random,
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random,                                               
                                                k_min=1, 
                                                k_max=kmax1)
    
    graphx_random1.update({"k_max": kmax1, "knn": "yes", "thresh": "no"})
    graphsRandom1.append(graphx_random1)
    
    ########
    avg_left_deg, avg_left_pos_deg, avg_left_neg_deg, avg_right_deg, avg_overlap, \
    avg_pos_overlap, avg_neg_overlap, only_pos, only_neg, empty_adj, unipos  = getconnectivity_info(graphx_random1["edges"], graphx_random1["labels"])

    graphsinfor.append({
        "Dataset (d)": "Productivity ("+ str(X.shape[1]) + ")",
        "kmax": kmax1, 
        "n": graphx_random1['n'],
        "m": graphx_random1['m'],
        "#+ves": len({t for t, l in graphx_random1['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random1['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg, 
        "avg LHS+d": avg_left_pos_deg, 
        "avg LHS-d": avg_left_neg_deg,  
        "avg RHSd": avg_right_deg,         
        "avg overlap": avg_overlap,
        "avg overlap+": avg_pos_overlap,
        "avg overlap-": avg_neg_overlap,
        "only+Ns": only_pos,
        "only-Ns": only_neg,
        "emptyNs": empty_adj,
        "uni+": unipos,   
        "graphID": graphid
    })
    
    graphid += 1

######
randomgraphsinfo = pd.DataFrame(graphsinfor)   

######
np.save("../graphs/productivity_knn_random.npy", np.array(graphsRandom1, dtype=object), allow_pickle=True)
randomgraphsinfo.to_csv("../graphs/productivity_knn_graphsummary.npy", index=False)
######
randomgraphsinfo
 


,Dataset (d),kmax,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Productivity (11),1,112,32,24,8,1.0,0.687500,0.312500,3.500000,0.040701,0.023488,0.017214,77,35,0,0,0
1,Productivity (11),2,112,44,33,11,2.0,1.455357,0.544643,5.090909,0.142375,0.096525,0.045849,62,11,0,0,1
2,Productivity (11),3,112,45,34,11,3.0,2.285714,0.714286,7.466667,0.294884,0.219916,0.074968,52,0,0,0,2
3,Productivity (11),4,112,46,35,11,4.0,2.946429,1.053571,9.739130,0.485199,0.335425,0.149775,28,0,0,0,3
4,Productivity (11),5,112,48,37,11,5.0,3.669643,1.330357,11.666667,0.738095,0.511422,0.226673,16,0,0,0,4
5,Productivity (11),6,112,48,37,11,6.0,4.401786,1.598214,14.000000,1.028958,0.712838,0.316120,9,0,0,0,5
6,Productivity (11),7,112,48,37,11,7.0,5.178571,1.821429,16.333333,1.351190,0.953668,0.397523,4,0,0,0,6
7,Productivity (11),8,112,48,37,11,8.0,6.017857,1.982143,18.666667,1.734878,1.267696,0.467181,3,0,0,0,7
8,Productivity (11),9,112,48,37,11,9.0,6.857143,2.142857,21.000000,2.141248,1.598295,0.542954,3,0,0,0,8
9,Productivity (11),10,112,48,37,11,10.0,7.642857,2.357143,23.333333,2.607143,1.949807,0.657336,2,0,0,0,9


In [8]:
#############
##thresh##
#############
graphsRandom11 = []
graphsinfor11 = []
graphid11 = 0
for t11 in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    _, _, graphx_random11 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random,
                                                sLHS=s_LHS_random,
                                                sRHS=s_RHS_random,                                        
                                                threshold=t11, 
                                                topk=False, 
                                                thresh=True)
    
    graphx_random11.update({"threshold": t11, "knn": "no", "thresh": "yes"})
    graphsRandom11.append(graphx_random11)

    ########
    avg_left_deg11, avg_left_pos_deg11, avg_left_neg_deg11, avg_right_deg11, avg_overlap11, \
    avg_pos_overlap11, avg_neg_overlap11, only_pos11, only_neg11, empty_adj11, unipos11  = getconnectivity_info(graphx_random11["edges"], graphx_random11["labels"])


    graphsinfor11.append({
        "Dataset (d)": "Productivity ("+ str(X.shape[1]) + ")",
        "r": t11, 
        "n": graphx_random11['n'],
        "m": graphx_random11['m'],
        "#+ves": len({t for t, l in graphx_random11['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random11['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg11, 
        "avg LHS+d": avg_left_pos_deg11, 
        "avg LHS-d": avg_left_neg_deg11,  
        "avg RHSd": avg_right_deg11,         
        "avg overlap": avg_overlap11,
        "avg overlap+": avg_pos_overlap11,
        "avg overlap-": avg_neg_overlap11,  
        "only+Ns": only_pos11,
        "only-Ns": only_neg11,
        "emptyNs": empty_adj11,
        "uni+": unipos11,
        "graphID": graphid11
    })
    
    graphid11 += 1

#######
randomgraphsinfo11 = pd.DataFrame(graphsinfor11)   
        
    
#######   
np.save("../graphs/productivity_thresh_random.npy", np.array(graphsRandom11, dtype=object), allow_pickle=True)
randomgraphsinfo11.to_csv("../graphs/productivity_thresh_graphsummary.npy", index=False)
#######
randomgraphsinfo11


,Dataset (d),r,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Productivity (11),4.0,112,50,39,11,20.330357,15.991071,4.339286,45.54,9.069088,7.081356,1.987732,0,1,7,0,0
1,Productivity (11),4.5,112,50,39,11,25.598214,19.982143,5.616071,57.34,14.040155,10.830350,3.209805,0,0,6,0,1
2,Productivity (11),5.0,112,50,39,11,32.116071,25.080357,7.035714,71.94,21.887921,17.002419,4.885502,0,0,6,0,2
3,Productivity (11),5.5,112,50,39,11,38.803571,30.339286,8.464286,86.92,31.333333,24.426060,6.907273,0,0,6,0,3
4,Productivity (11),6.0,112,50,39,11,44.241071,34.562500,9.678571,99.10,39.637286,30.956494,8.680793,0,0,5,1,4
5,Productivity (11),6.5,112,50,39,11,46.696429,36.419643,10.276786,104.60,43.723010,34.105221,9.617789,0,0,5,13,5
6,Productivity (11),7.0,112,50,39,11,47.491071,37.035714,10.455357,106.38,45.163874,35.214631,9.949243,0,0,5,25,6
7,Productivity (11),7.5,112,50,39,11,47.732143,37.223214,10.508929,106.92,45.621334,35.569610,10.051724,0,0,5,35,7
8,Productivity (11),8.0,112,50,39,11,47.767857,37.258929,10.508929,107.00,45.689655,35.637931,10.051724,0,0,5,39,8
9,Productivity (11),8.5,112,50,39,11,47.839286,37.303571,10.535714,107.16,45.798068,35.701127,10.096940,0,0,4,5,9
